# Step 2. Preprocessing data in h5 file: remove outliers, adjust amplitudes.

In [1]:
import numpy as np
import cupy as cp
import h5py
import matplotlib.pyplot as plt
import cupyx.scipy.ndimage as ndimage
from holotomocupy.utils import *


# Init

In [2]:
ntheta = 1800
ids = np.arange(0,1800,1800/ntheta).astype('int')



path_out = '/data2/vnikitin/atomium_rec/20250607/AtomiumL1'
file_out = f'data.h5'

with h5py.File(f'{path_out}/{file_out}') as fid:
    detector_pixelsize = fid['/exchange/detector_pixelsize'][0]    
    focustodetectordistance = fid['/exchange/focusdetectordistance'][0]    
    z1 = fid['/exchange/z1'][:] 
    
    shifts = fid['/exchange/shifts'][ids]
    attrs = fid['/exchange/attrs'][ids]
    shape = np.array(fid[f'/exchange/data0'].shape)
    shape_ref = fid['/exchange/data_white_start0'].shape
    shape_dark = fid['/exchange/data_dark0'].shape    
    

## sizes
n = shape[1]
ndark = shape_dark[0]
nref = shape_ref[0]
ndist = len(z1)
print(f'{z1=}')
print(f'{focustodetectordistance=}')
print(f'{detector_pixelsize=}')


z1=array([0.006113, 0.006375, 0.007424, 0.009602], dtype=float32)
focustodetectordistance=np.float32(1.2889996)
detector_pixelsize=np.float32(2.9520295e-06)


### Remove outliers function

In [3]:
def remove_outliers(data, dezinger, dezinger_threshold):    
    res = data.copy()
    w = [dezinger,dezinger]
    for k in range(data.shape[0]):
        data0 = cp.array(data[k])
        fdata = ndimage.median_filter(data0, w)
        # print(np.sum(np.abs(data0-fdata)>fdata*dezinger_threshold))
        res[k] = np.where(np.abs(data0-fdata)>fdata*dezinger_threshold, fdata, data0).get()
    return res

### Read ref and dark

In [4]:
ref0 = np.empty([nref,ndist,n,n],dtype='float32')
ref1 = np.empty([nref,ndist,n,n],dtype='float32')
dark = np.empty([ndark,ndist,n,n],dtype='float32')
with h5py.File(f'{path_out}/{file_out}') as fid:
    for k in range(ndist):
        ref0[:,k] = fid[f'/exchange/data_white_start{k}'][:]
        ref1[:,k] = fid[f'/exchange/data_white_end{k}'][:]
        dark[:,k] = fid[f'/exchange/data_dark{k}'][:]
        

### Remove outliers

In [5]:
ref = ref0#(ref0+ref1)*0.5
dark = np.mean(dark,axis=0)
ref = np.mean(ref,axis=0)
ref-=dark
ref[ref<0]=0
radius = 3
threshold = 0.9
ref[:] = remove_outliers(ref[:], radius, threshold)     

#### Normalization

In [6]:
mean_data_ref = np.zeros(ndist,dtype='float32')
with h5py.File(f'{path_out}/{file_out}','a') as fid:    
    for k in range(ndist):
        if f'/exchange/pdata{k}' in fid:
            del fid[f'/exchange/pdata{k}']
        data_out = fid.create_dataset(f'/exchange/pdata{k}', shape = shape)   
        for j in range(1):
            data = fid[f'/exchange/data{k}'][ids[j]].astype('float32')
            data-=dark[k]
            data[data<0]=0
            data = remove_outliers(data[None], radius, threshold)[0]
            mean_data_ref[k] = np.mean(data)
            

In [7]:
### counts before scan
mmr = np.mean(ref,axis=(1,2))
# scale mean of first projection based on that
mean_data_ref *= mmr[0]/mmr[:]
# scale ref based on that
ref *= mmr[0]/mmr[:,None,None]

mean_data_ref/=mmr[0]
ref/=mmr[0]

# ref /= mean_data_ref[:,None,None]
print(np.mean(ref,axis=(1,2)))
# ss


[1.0000004  0.9999999  0.99999976 1.0000004 ]


In [8]:
with h5py.File(f'{path_out}/{file_out}','a') as fid:    
    if f'/exchange/pref' in fid:
        del fid[f'/exchange/pref']
    fid.create_dataset(f'/exchange/pref', data=ref)
    

### Process data 

In [ ]:
with h5py.File(f'{path_out}/{file_out}','a') as fid:    
    for k in range(ndist):
        if f'/exchange/pdata{k}' in fid:
            del fid[f'/exchange/pdata{k}']
        data_out = fid.create_dataset(f'/exchange/pdata{k}', shape = shape)   
        for j in range(ntheta):
            data = fid[f'/exchange/data{k}'][ids[j]].astype('float32')
            data-=dark[k]
            data[data<0]=0
            data = remove_outliers(data[None], radius, threshold)[0]
            data=data/np.mean(data)*mean_data_ref[k]
            data_out[j] = data
            if j%100==0:
                print(j,k,np.mean(data))

0 0 1.0023332
100 0 1.0023324
200 0 1.0023324
300 0 1.0023335
400 0 1.0023334
500 0 1.0023335
600 0 1.0023333
700 0 1.0023321
800 0 1.0023327
900 0 1.0023328
1000 0 1.0023338
1100 0 1.0023329
1200 0 1.0023334
1300 0 1.0023333
1400 0 1.0023324
1500 0 1.0023324
1600 0 1.0023328
1700 0 1.0023339
0 1 1.0001522
100 1 1.0001513
200 1 1.0001518
300 1 1.0001507
400 1 1.0001519
500 1 1.000152
600 1 1.0001528
700 1 1.0001519
800 1 1.0001525
900 1 1.0001521
1000 1 1.0001527
1100 1 1.000152
1200 1 1.0001522
1300 1 1.0001525
1400 1 1.0001519
1500 1 1.0001525
1600 1 1.000152
1700 1 1.000152
0 2 0.9991222
100 2 0.9991221
200 2 0.99912244
300 2 0.99912244
400 2 0.9991232
500 2 0.99912286
600 2 0.99912214
700 2 0.999123
800 2 0.9991218
900 2 0.9991218
1000 2 0.9991224
1100 2 0.9991217
1200 2 0.99912244
1300 2 0.9991225
1400 2 0.99912155
1500 2 0.9991217
1600 2 0.9991235
1700 2 0.99912155
0 3 1.0059974
100 3 1.0059984
200 3 1.0059971
300 3 1.0059974
400 3 1.0059974
500 3 1.0059988
600 3 1.0059971
700 3 

: 